# Tutorial: rediscover 6J5W in EMD-0683

[6J5W](https://www.rcsb.org/structure/6J5W) is an Arabidopsis thaliana ZAR1–RKS1
complex fitted to the 3.7 Å map [EMD-0683](https://www.ebi.ac.uk/emdb/EMD-0683).
We treat ZAR1 (chain A, UniProt Q38834) as the unknown protein and RKS1 (chain
B) as its known partner. We remove the RKS1 density from the map, fit candidate
structures into the remaining density with PowerFit, and refine the fits against
RKS1 with HADDOCK3.

Protein-detective is intended for low-resolution maps (about 6–15 Å). EMD-0683
is a high-resolution map (<4.5 Å), used here because its deposited model lets us
check the results.

![EMD-0683 density with the fitted 6J5W model](EMD-0683.png)

EMD-0683 with the fitted 6J5W model: ZAR1 (orange, bottom) is the protein to
rediscover, RKS1 (green, top) is the known partner.

<!--
Screenshot from the
[EMDB 3D viewer](https://www.ebi.ac.uk/emdb/EMD-0683?tab=3dview)
with reduced volume opacity.
-->

## 1. Requirements

- [protein-detective](index.md) on your `PATH` (Python 3.13 or later)
- `curl`, `gzip`, and the [DuckDB CLI](https://duckdb.org/docs/stable/clients/cli/overview)
- [ChimeraX](https://www.rbvi.ucsf.edu/chimerax/), run as Flatpak by default;
  set the `CHIMERAX` environment variable (for example `CHIMERAX=chimerax`) for
  a native install
- a GPU for PowerFit

All files go into the session directory `tutorial-6j5w/`, next to this notebook.

## 2. Search for candidates

Search reviewed Arabidopsis proteins annotated with ADP binding
([GO:0043531](https://www.ebi.ac.uk/QuickGO/term/GO:0043531)), with canonical
sequences of 800–900 residues. Include up to ten AlphaFold models alongside
the PDBe structures to provide 5–15 candidates after filtering.

In [1]:
%%bash
set -euo pipefail

protein-detective candidates search tutorial-6j5w \
    --taxon-id 3702 \
    --reviewed \
    --molecular-function-go GO:0043531 \
    --min-sequence-length 800 \
    --max-sequence-length 900 \
    --limit-uniprot 100 \
    --alphafold.limit 10 \
    --pdbe.limit 50 \
    --pdbe.min-residues 750 \
    --pdbe.max-residues 900 \
    --pdbe.top-resolution-per-uniprot-accession 2

Searching for UniProt accessions


Found 27 UniProt accessions, written to tutorial-6j5w/uniprot.txt


Finding AlphaFold entries for 27 uniprot accessions


Search for alphafold entries on uniprot returned 10 results. There may be more results available, bu

t they are not returned due to the limit of 10. Consider increasing the limit to get more results.


...


Searching for quality scores for PDBe entries 100% ━━━ 4/4  [ 0:… < 0:0… , ? 

  ]
                                                                           ac…  


Searching for quality scores for PDBe entries 100% ━━━ 4/4  [ 0:… < 0:0… , ? 

  ]
                                                                           ac…  


Written quality scores for 4 PDBe entries to tutorial-6j5w/pdbe-quality.json


In [2]:
import pandas as pd

pdbe = pd.read_csv("tutorial-6j5w/pdbe.csv")
pdbe

,uniprot_accession,pdb_id,method,resolution,uniprot_chains,chain,chain_length
0,Q38834,6J5T,Electron_Microscopy,3.4,C/F/G/L/O=1-852,C,852
1,Q38834,6J5W,Electron_Microscopy,3.7,A=1-852,A,852
2,Q9FKZ1,8YL6,Electron_Microscopy,3.1,C=1-809,C,809
3,Q9LZ25,9JBN,Electron_Microscopy,2.3,C=1-811,C,811


Check that 6J5W was found:

In [3]:
assert (pdbe["pdb_id"] == "6J5W").any(), "The search must include 6J5W"

## 3. Retrieve and filter the structures

In [4]:
%%bash
set -euo pipefail

protein-detective candidates retrieve tutorial-6j5w

protein-detective candidates filter tutorial-6j5w \
    --min-residues 750 \
    --max-residues 900 \
    --min-geometry-quality 50 \
    --top-uniprot-cluster 2 \
    --scheduler-address sequential

ls tutorial-6j5w/combined_output

Retrieving 4 PDBe entries


Retrieved 4 PDBe entries, written to tutorial-6j5w/downloads/pdbe


Statistics written to tutorial-6j5w/pdbe_stats.csv


Retrieving 10 AlphaFold entries with formats {'cif'}


Retrieved 10 AlphaFold files and 0 summaries, written to 
tutorial-6j5w/down

...

Statistics written to tutorial-6j5w/combined_stats.csv


6j5t_updated_C2A.cif.gz
6j5w_updated_A2A.cif.gz
AF-O64973-F1-model_v6.cif.gz
AF-O82484-F1-model_v6.c

if.gz
AF-P60838-F1-model_v6.cif.gz
AF-P60839-F1-model_v6.cif.gz
AF-Q38834-F1-model_v6.cif.gz
AF-Q8L3

R3-F1-model_v6.cif.gz
AF-Q8RXS5-F1-model_v6.cif.gz
AF-Q940K0-F1-model_v6.cif.gz
AF-Q9C646-F1-model_v

6.cif.gz
AF-Q9C8K0-F1-model_v6.cif.gz


In [5]:
stats = pd.read_csv("tutorial-6j5w/combined_stats.csv")
candidates = stats.loc[stats["passed"]]
assert 5 <= len(candidates) <= 15, f"Expected 5-15 candidate structures, found {len(candidates)}"
assert (candidates["structure_id"] == "6J5W").any(), "6J5W must pass the filters"
stats

,input_file,structure_id,uniprot_accession,resolution,high_confidence_residues_count,total_residue_count,method,is_alphafold,uniprot_start,uniprot_end,sequence_identity,chain_length,geometry_quality,passed,output_file,reason
0,combined_input/8yl6_updated_C2A.cif.gz,8YL6,Q9FKZ1,3.1,NaN,397,EM,False,1,809,1.0,397,71.82,False,NaN,"Chain length 397 not in range [750, 900]"
1,combined_input/9jbn_updated_C2A.cif.gz,9JBN,Q9LZ25,2.3,NaN,423,EM,False,1,811,1.0,423,61.40,False,NaN,"Chain length 423 not in range [750, 900]"
2,combined_input/AF-O64973-F1-model_v6.cif.gz,AF-O64973-F1,O64973,0.0,804.0,889,Predicted,True,1,889,1.0,889,NaN,True,combined_output/AF-O64973-F1-model_v6.cif.gz,NaN
3,combined_input/AF-O82484-F1-model_v6.cif.gz,AF-O82484-F1,O82484,0.0,791.0,892,Predicted,True,1,892,1.0,892,NaN,True,combined_output/AF-O82484-F1-model_v6.cif.gz,NaN
4,combined_input/AF-P60838-F1-model_v6.cif.gz,AF-P60838-F1,P60838,0.0,808.0,894,Predicted,True,1,894,1.0,894,NaN,True,combined_output/AF-P60838-F1-model_v6.cif.gz,NaN
5,combined_input/AF-P60839-F1-model_v6.cif.gz,AF-P60839-F1,P60839,0.0,797.0,884,Predicted,True,1,884,1.0,884,NaN,True,combined_output/AF-P60839-F1-model_v6.cif.gz,NaN
6,combined_input/AF-Q38834-F1-model_v6.cif.gz,AF-Q38834-F1,Q38834,0.0,766.0,852,Predicted,True,1,852,1.0,852,NaN,True,combined_output/AF-Q38834-F1-model_v6.cif.gz,NaN
7,combined_input/AF-Q8L3R3-F1-model_v6.cif.gz,AF-Q8L3R3-F1,Q8L3R3,0.0,810.0,885,Predicted,True,1,885,1.0,885,NaN,True,combined_output/AF-Q8L3R3-F1-model_v6.cif.gz,NaN
8,combined_input/AF-Q8RXS5-F1-model_v6.cif.gz,AF-Q8RXS5-F1,Q8RXS5,0.0,805.0,888,Predicted,True,1,888,1.0,888,NaN,True,combined_output/AF-Q8RXS5-F1-model_v6.cif.gz,NaN
9,combined_input/AF-Q940K0-F1-model_v6.cif.gz,AF-Q940K0-F1,Q940K0,0.0,791.0,889,Predicted,True,1,889,1.0,889,NaN,True,combined_output/AF-Q940K0-F1-model_v6.cif.gz,NaN


Twelve structures pass: ten AlphaFold models and two PDBe structures of ZAR1,
6J5T (`6j5t_updated_C2A.cif.gz`) and 6J5W (`6j5w_updated_A2A.cif.gz`).
The other PDBe candidates have too few residues. Counts can change as the
databases are updated.

## 4. Download the experimental map and reference complex


In [6]:
%%bash
set -euo pipefail

curl -fsSL --retry 3 \
    https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-0683/map/emd_0683.map.gz \
    -o tutorial-6j5w/emd_0683.map.gz

gzip -dk tutorial-6j5w/emd_0683.map.gz

curl -fsSL --retry 3 \
    https://www.ebi.ac.uk/pdbe/entry-files/download/6j5w_updated.cif.gz \
    -o tutorial-6j5w/6j5w_updated.cif.gz

## 5. Remove the known RKS1 density from the map

Simulate a 3.7 Å map of RKS1 and zero the experimental map inside it, as in
[the EM preparation notebook](em_prepare.ipynb).

In [7]:
%%bash
set -euo pipefail

gzip -dk tutorial-6j5w/6j5w_updated.cif.gz

cat > tutorial-6j5w/mask-rks1.cxc <<'EOF'
open emd_0683.map
open 6j5w_updated.cif
delete #2/A
molmap #2 3.7
volume mask #1 surfaces #3 invertMask true
save emd_0683-without-rks1.mrc models #4
exit
EOF

(cd tutorial-6j5w &&
    ${CHIMERAX:-flatpak run edu.ucsf.rbvi.ChimeraX} --nogui --exit --script mask-rks1.cxc)

INFO:
Executing: runscript mask-rks1.cxc


INFO:
Executing: open emd_0683.map


INFO:
Opened emd_0683.map as #1, grid size 160,160,160, pixel 1.09, shown at level 0.0364, step 1, v

alues float32


INFO:
Executing: open 6j5w_updated.cif


...

loat32


INFO:
Executing: save emd_0683-without-rks1.mrc models #4


STATUS:
Wrote file emd_0683-without-rks1.mrc


INFO:
Executing: exit


STATUS:
Exiting ...


Compare the density before and after masking: ZAR1 (chain A) should retain about 100%, while RKS1 (chain B) should
retain 0%.

In [8]:
from pathlib import Path

import gemmi
import numpy as np

session = Path("tutorial-6j5w")
structure = gemmi.read_structure(str(session / "6j5w_updated.cif"))
original = gemmi.read_ccp4_map(str(session / "emd_0683.map"), setup=True).grid
masked = gemmi.read_ccp4_map(str(session / "emd_0683-without-rks1.mrc"), setup=True).grid
rows = []
for chain_id, protein in [("A", "ZAR1"), ("B", "RKS1")]:
    positions = [atom.pos for residue in structure[0][chain_id] for atom in residue if atom.name == "CA"]
    before = np.mean([original.interpolate_value(pos) for pos in positions])
    after = np.mean([masked.interpolate_value(pos) for pos in positions])
    rows.append(
        {
            "Protein": protein,
            "Chain": chain_id,
            "Before masking": before,
            "After masking": after,
            "Density retained (%)": 100 * after / before,
        }
    )
pd.DataFrame(rows).round(3)

,Protein,Chain,Before masking,After masking,Density retained (%)
0,ZAR1,A,0.082,0.082,100.0
1,RKS1,B,0.076,0.000,0.0


## 6. Fit the candidates into the masked EMD-0683 map

Fit each candidate with PowerFit. Runtime depends on the number of candidates
and the GPU.


In [9]:
%%bash
set -euo pipefail

protein-detective powerfit run \
    tutorial-6j5w/emd_0683-without-rks1.mrc 3.7 \
    tutorial-6j5w \
    --angle 10 \
    --scheduler-address sequential \
    --powerfit-run-id emd0683

protein-detective powerfit list-runs tutorial-6j5w \
    --output tutorial-6j5w/powerfit-runs.csv

protein-detective powerfit report tutorial-6j5w \
    --powerfit-run-id emd0683 \
    --top 1 \
    --output tutorial-6j5w/powerfit-report.csv


Running PowerFit sequentially:   0%|          | 0/12 [00:00<?, ?structure/s]


Running PowerFit sequentially:   8%|▊         | 1/12 [00:12<02:15, 12.28s/structure]


Running PowerFit sequentially:  17%|█▋        | 2/12 [00:24<02:00, 12.09s/structure]


Running PowerFit sequentially:  25%|██▌       | 3/12 [00:36<01:48, 12.02s/structure]


Running PowerFit sequentially:  33%|███▎      | 4/12 [00:48<01:35, 11.98s/structure]

...


Running PowerFit sequentially: 100%|██████████| 12/12 [02:23<00:00, 11.95s/stru

cture]


Running PowerFit sequentially: 100%|██████████| 12/12 [02:23<00:00, 11.97s/stru

cture]


PowerFit run completed with ID: emd0683. Use this ID for reporting or fitting 
models.


In [10]:
pd.read_csv("tutorial-6j5w/powerfit-report.csv")

,powerfit_run_id,structure,rank,cc,fishz,relz,translation,rotation,template_file,uniprot_accessions,structure_id,is_alphafold
0,emd0683,6j5w_updated_A2A.cif.gz,1,0.310,0.320,44.290,87.28:90.553:97.099,1.0:0.0:0.0:0.0:1.0:0.0:0.0:0.0:1.0,tutorial-6j5w/combined_output/6j5w_updated_A2A...,Q38834,6J5W,False
1,emd0683,AF-Q8RXS5-F1-model_v6.cif.gz,1,0.092,0.092,12.765,82.916:104.736:97.099,-0.087:0.958:-0.273:-0.818:0.087:0.568:0.568:0...,tutorial-6j5w/combined_output/AF-Q8RXS5-F1-mod...,Q8RXS5,AF-Q8RXS5-F1,True
2,emd0683,AF-O82484-F1-model_v6.cif.gz,1,0.091,0.091,12.487,85.098:104.736:97.099,-0.353:0.918:-0.182:-0.781:-0.182:0.597:0.515:...,tutorial-6j5w/combined_output/AF-O82484-F1-mod...,O82484,AF-O82484-F1,True
3,emd0683,AF-P60838-F1-model_v6.cif.gz,1,0.080,0.080,11.095,99.281:98.19:92.735,-1.0:0.0:0.0:0.0:-0.3:0.954:0.0:0.954:0.3,tutorial-6j5w/combined_output/AF-P60838-F1-mod...,P60838,AF-P60838-F1,True
4,emd0683,AF-P60839-F1-model_v6.cif.gz,1,0.074,0.074,10.178,87.28:104.736:97.099,-0.273:0.958:0.087:-0.776:-0.273:0.568:0.568:0...,tutorial-6j5w/combined_output/AF-P60839-F1-mod...,P60839,AF-P60839-F1,True
5,emd0683,AF-O64973-F1-model_v6.cif.gz,1,0.069,0.069,9.594,81.825:102.554:98.19,0.0:1.0:-0.0:-0.815:0.0:0.58:0.58:-0.0:0.815,tutorial-6j5w/combined_output/AF-O64973-F1-mod...,O64973,AF-O64973-F1,True
6,emd0683,AF-Q940K0-F1-model_v6.cif.gz,1,0.068,0.068,9.376,64.369:101.463:101.463,0.901:0.33:-0.28:-0.033:0.698:0.715:0.432:-0.6...,tutorial-6j5w/combined_output/AF-Q940K0-F1-mod...,Q940K0,AF-Q940K0-F1,True
7,emd0683,AF-Q38834-F1-model_v6.cif.gz,1,0.067,0.068,9.089,86.189:90.553:96.008,0.17:0.83:0.531:-0.83:-0.17:0.531:0.531:-0.531...,tutorial-6j5w/combined_output/AF-Q38834-F1-mod...,Q38834,AF-Q38834-F1,True
8,emd0683,6j5t_updated_C2A.cif.gz,1,0.066,0.066,9.082,85.098:92.735:105.827,-0.087:0.568:0.818:-0.273:0.776:-0.568:-0.958:...,tutorial-6j5w/combined_output/6j5t_updated_C2A...,Q38834,6J5T,False
9,emd0683,AF-Q9C8K0-F1-model_v6.cif.gz,1,0.066,0.067,8.923,67.642:113.464:98.19,0.33:0.901:-0.28:-0.635:0.432:0.64:0.698:-0.03...,tutorial-6j5w/combined_output/AF-Q9C8K0-F1-mod...,Q9C8K0,AF-Q9C8K0-F1,True


The report has the best solution per candidate; higher `cc` is better. 6J5W
scores 0.31 in its deposited pose, 6J5T only 0.066.

Write the fitted models:

In [11]:
%%bash
set -euo pipefail

protein-detective powerfit fit-models tutorial-6j5w \
    --powerfit-run-id emd0683 \
    --top 1 \
    --output tutorial-6j5w/fitted-models.csv


Writing fitted model PDB files:   0%|          | 0/12 [00:00<?, ?it/s]


Writing fitted model PDB files:  33%|███▎      | 4/12 [00:00<00:00, 31.14it/s]


Writing fitted model PDB files:  67%|██████▋   | 8/12 [00:00<00:00, 31.56it/s]


Writing fitted model PDB files: 100%|██████████| 12/12 [00:00<00:00, 31.74it/s]


Writing fitted model PDB files: 100%|██████████| 12/12 [00:00<00:00, 31.64it/s]

## 7. Refine against the known RKS1 partner

Use 6J5W without chain A as the known RKS1 partner. Use ten rigid-body samples
and refine the best model from the top cluster with four refinement samples. Refining all twelve candidates takes tens of
minutes on a six-core CPU.

In [12]:
%%bash
set -euo pipefail

protein-detective refine run \
    tutorial-6j5w \
    tutorial-6j5w/6j5w_updated.cif.gz \
    --remove-known-chains A \
    --powerfit-run-id emd0683 \
    --refine-run-id emd0683-small \
    --rigidbody-sampling 10 \
    --top-clusters 1 \
    --top-models 1 \
    --water-refinement-sampling-factor 4 \
    --water-refinement-solvent none \
    --ncores "$(nproc)" \
    --scheduler-address sequential

protein-detective refine list-runs tutorial-6j5w \
    --output tutorial-6j5w/refine-runs.csv

protein-detective refine report tutorial-6j5w \
    --refine-run-id emd0683-small \
    --top 1 \
    --output tutorial-6j5w/refined.csv


Refined structures:   0%|          | 0/12 [00:00<?, ?file/s]

[2026-10-17 18:37:18,580 libclust WARNING] Reordering by cluster rank and model rank


[2026-10-17 18:38:05,597 libclust WARNING] Reordering by cluster rank and model rank



Refined structures:   8%|▊         | 1/12 [03:16<36:02, 196.62s/file]

[2026-10-17 18:40:08,593 libclust WARNING] Reordering by cluster rank and model rank


...

[2026-10-17 19:08:43,852 libclust WARNING] Reordering by cluster rank and model rank


[2026-10-17 19:09:32,149 libclust WARNING] Reordering by cluster rank and model rank



Refined structures: 100%|██████████| 12/12 [34:42<00:00, 171.82s/file]


Refined structures: 100%|██████████| 12/12 [34:42<00:00, 173.58s/file]


Refinement run completed with ID: emd0683-small


In [13]:
display(pd.read_csv("tutorial-6j5w/refine-runs.csv"))
pd.read_csv("tutorial-6j5w/refined.csv")

,refine_run_id,run_dir,known_structure,options
0,emd0683-small,tutorial-6j5w/refine/emd0683-small,/home/verhoes/git/protein-detective/protein-de...,--refine-run-id emd0683-small --rigidbody-samp...


,refine_run_id,powerfit_run_id,structure,fitted_rank,uniprot_accessions,cluster_id,cluster_rank,population,under_eval,cluster_score,cluster_score_std,modelcluster_ranking,unrefined_score,refined_score,refined_structure_path,fitted_model,refine_run_dir
0,emd0683-small,emd0683,6j5t_updated_C2A.cif.gz,1,Q38834,1,1,4,NaN,-93.427,3.345,1,-30.430,-96.929,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...,tutorial-6j5w/powerfit/emd0683/6j5t_updated_C2...,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...
1,emd0683-small,emd0683,6j5w_updated_A2A.cif.gz,1,Q38834,1,1,4,NaN,-85.298,7.907,1,-258.006,-94.928,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...,tutorial-6j5w/powerfit/emd0683/6j5w_updated_A2...,tutorial-6j5w/refine/emd0683-small/emd0683/6j5...
2,emd0683-small,emd0683,AF-O64973-F1-model_v6.cif.gz,1,O64973,1,1,4,NaN,-111.747,6.227,1,94.703,-119.262,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-O64973-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...
3,emd0683-small,emd0683,AF-O82484-F1-model_v6.cif.gz,1,O82484,1,1,4,NaN,-104.983,5.435,1,54.342,-111.761,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-O82484-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...
4,emd0683-small,emd0683,AF-P60838-F1-model_v6.cif.gz,1,P60838,1,1,4,NaN,-188.334,13.714,1,5177.057,-210.124,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-P60838-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...
5,emd0683-small,emd0683,AF-P60839-F1-model_v6.cif.gz,1,P60839,1,1,4,NaN,-66.193,7.117,1,-21.004,-78.226,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-P60839-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...
6,emd0683-small,emd0683,AF-Q38834-F1-model_v6.cif.gz,1,Q38834,1,1,4,NaN,-118.710,3.002,1,1510.688,-123.240,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-Q38834-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...
7,emd0683-small,emd0683,AF-Q8L3R3-F1-model_v6.cif.gz,1,Q8L3R3,1,1,4,NaN,-61.814,2.010,1,1980.835,-64.119,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-Q8L3R3-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...
8,emd0683-small,emd0683,AF-Q8RXS5-F1-model_v6.cif.gz,1,Q8RXS5,1,1,4,NaN,-47.459,3.592,1,166.332,-52.834,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-Q8RXS5-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...
9,emd0683-small,emd0683,AF-Q940K0-F1-model_v6.cif.gz,1,Q940K0,1,1,4,NaN,-60.881,7.590,1,1.506,-71.108,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...,tutorial-6j5w/powerfit/emd0683/AF-Q940K0-F1-mo...,tutorial-6j5w/refine/emd0683-small/emd0683/AF-...


Lower HADDOCK scores are better. Compare baseline and refined scores in the
table above. Ten rigid-body samples are only a quick demonstration and can
lose the native pose, so increase `--rigidbody-sampling` for real use.

## 8. Visualize fitted models

<!--
TODO: add a `protein-detective visualize` CLI command that shows the fitted
(PowerFit) and refined (HADDOCK3) structures together with the density map,
so this chapter can use a single command instead of the Python script below.
-->

Write an HTML viewer of the fitted models and the masked map:

In [14]:
from pathlib import Path

from protein_detective.visualization import (
    load_fitted_models,
    show_fitted_models_and_density,
)

root = Path("tutorial-6j5w")
models = load_fitted_models(
    fitted_models_csv=root / "fitted-models.csv",
    report_csv=root / "powerfit-report.csv",
    runs_csv=root / "powerfit-runs.csv",
)
html = show_fitted_models_and_density(models, renderer="html")
(root / "fitted-models.html").write_text(html)

30150112

Open `tutorial-6j5w/fitted-models.html` in a browser. See
[the visualization notebook](visualization.ipynb) for more examples.

## 9. Build the metadata database and check the results

Compare baseline and refined scores for a few models:
Paths in the metadata tables are relative to `tutorial-6j5w/`.


In [15]:
%%bash
set -euo pipefail

protein-detective meta tutorial-6j5w

tutorial-6j5w/meta.duckdb has been generated.
See https://www.bonvinlab.org/protein-dete

ctive/meta.html for example queries.
Use `duckdb --readonly tutorial-6j5w/meta.duckdb` t

o query it.


In [16]:
%%bash
duckdb --readonly tutorial-6j5w/meta.duckdb -c "
SELECT
    u.score AS unrefined_score,
    m.score AS refined_score,
    m.refined_structure_path
FROM refinements_capri_ss m
JOIN refinements_baseline u USING (refine_run_dir)
ORDER BY refined_score
LIMIT 5;
"


┌─────────────────┬───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ unrefined_score │ refined_score │                                        refined_structure_path                                         │
│     double      │    double     │                                                varchar                                                │
├─────────────────┼───────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        5177.057 │      -210.124 │ refine/emd0683-small/emd0683/AF-P60838-F1-model_v6.cif.gz/fit_1.pdb/refinement/6_mdref/mdref_3.pdb.gz │
│        5177.057 │      -188.556 │ refine/emd0683-small/emd0683/AF-P60838-F1-model_v6.cif.gz/fit_1.pdb/refinement/6_mdref/mdref_1.pdb.gz │
│        5177.057 │      -181.518 │ refine/emd0683-small/emd0683/AF-P60838-F1-model_v6.cif.gz/fit_1.pdb/refinement/6_mdref/mdref_4.pdb.gz │
│        5177.057 │ 

See [the metadata notebook](meta.ipynb) for queries on `meta.duckdb`.